# Taller 3 - MLOps: Pipeline de scikit-learn con MLflow en Databricks

Link del repositorio: https://github.com/faguilarleal/taller3_mlops

Integrantes
- Francis Aguilar
- Paula Barillas
- Bianca Calderon
- Monica Salvatierra
- Gerardo Pineda
- Jose Marchena

## Introducción


| Funcionalidad | Para qué la usamos |
|---|---|
| **Experiments** (`start_run`, `log_params`, `log_metrics`, `log_figure`) | Guardar cada entrenamiento con sus hiperparámetros, métricas y gráficas para compararlos en la UI. |
| **Autologging** (`mlflow.sklearn.autolog`) | Registrar parámetros y modelo automáticamente y contrastarlo con el logging manual. |
| **Models** (`log_model` con `signature` e `input_example`) | Guardar el pipeline completo (scaler + modelo) con el esquema de entrada/salida. |
| **Búsqueda de runs** (`search_runs`) | Elegir el mejor modelo con código, no a mano. |
| **Model Registry en Unity Catalog** (`register_model`, alias `champion`) | Versionar el modelo elegido y cargarlo por alias para usarlo en producción. |
| **Feature Store** (opcional) | Guardar las variables en una tabla gobernada y reutilizable. |

## 1. Dependencias

## 2. Imports y configuración


In [2]:
import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

EXPERIMENT_PATH = "/Users/pin22880@uvg.edu.gt/taller3_mlops"  
CATALOGO = "workspace"  
ESQUEMA = "default"   

MODEL_NAME = f"{CATALOGO}.{ESQUEMA}.taller3_mlops_model"
METRICA_PRINCIPAL = "test_f1"
TAG_PROYECTO = "taller3_mlops" 

mlflow.set_experiment(EXPERIMENT_PATH)
mlflow.set_registry_uri("databricks-uc")

print("Experimento:", EXPERIMENT_PATH)
print("Modelo:", MODEL_NAME)

2026/10/16 21:59:38 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/16 21:59:38 INFO mlflow.store.db.utils: Updating database tables


2026/10/16 21:59:42 INFO mlflow.tracking.fluent: Experiment with name '/Users/pin22880@uvg.edu.gt/taller3_mlops' does not exist. Creating a new experiment.


[local] set_registry_uri('databricks-uc') ignorado
Experimento: /Users/pin22880@uvg.edu.gt/taller3_mlops
Modelo: workspace.default.taller3_mlops_model


## 3. Carga de datos y split

In [3]:
X, y = make_classification(
    n_samples=1200,
    n_features=12,
    n_informative=8,
    n_redundant=2,
    n_classes=2,
    weights=[0.7, 0.3],
    flip_y=0.04,
    class_sep=1.2,
    random_state=42,
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

feature_names = [f"feature_{i}" for i in range(X.shape[1])]
X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_test_df = pd.DataFrame(X_test, columns=feature_names)

print("Forma de los datos:", X.shape)
print("Distribución objetivo:", pd.Series(y).value_counts().to_dict())

Forma de los datos: (1200, 12)
Distribución objetivo: {0: 836, 1: 364}


## 4. Funciones auxiliares

In [4]:
def calcular_metricas(modelo, X, y):
    y_pred = modelo.predict(X)
    y_proba = modelo.predict_proba(X)[:, 1]
    return {
        "test_accuracy": accuracy_score(y, y_pred),
        "test_precision": precision_score(y, y_pred, zero_division=0),
        "test_recall": recall_score(y, y_pred, zero_division=0),
        "test_f1": f1_score(y, y_pred, zero_division=0),
        "test_roc_auc": roc_auc_score(y, y_proba),
        "test_balanced_accuracy": balanced_accuracy_score(y, y_pred),
        "test_log_loss": log_loss(y, y_proba),
    }

def figura_matriz_confusion(modelo, X, y, titulo):
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_estimator(modelo, X, y, ax=ax, colorbar=False)
    ax.set_title(titulo)
    fig.tight_layout()
    return fig

def figura_importancia(modelo, feature_names, titulo):
    estimador = modelo.named_steps["model"]
    if hasattr(estimador, "coef_"):
        valores, etiqueta = estimador.coef_[0], "Coeficiente"
    else:
        valores, etiqueta = estimador.feature_importances_, "Importancia"
    serie = pd.Series(valores, index=feature_names).sort_values()
    fig, ax = plt.subplots(figsize=(6, 4))
    serie.plot.barh(ax=ax)
    ax.set_xlabel(etiqueta)
    ax.set_title(titulo)
    fig.tight_layout()
    return fig

## 5. EXPERIMENTS

In [5]:
def entrenar_run_manual(run_name, pipeline):
    with mlflow.start_run(run_name=run_name) as run:
        pipeline.fit(X_train_df, y_train)
        metricas = calcular_metricas(pipeline, X_test_df, y_test)

        estimador = pipeline.named_steps["model"]
        params = {f"model__{k}": v for k, v in estimador.get_params().items()}
        params["modelo"] = type(estimador).__name__
        params["scaler"] = type(pipeline.named_steps["scaler"]).__name__
        mlflow.log_params(params)
        mlflow.log_metrics(metricas)
        mlflow.set_tags({"proyecto": TAG_PROYECTO, "tipo_logging": "manual"})

        for fig, ruta in [
            (figura_matriz_confusion(pipeline, X_test_df, y_test, run_name), "graficas/matriz_confusion.png"),
            (figura_importancia(pipeline, feature_names, run_name), "graficas/importancia_variables.png"),
        ]:
            mlflow.log_figure(fig, ruta)
            plt.close(fig)

        signature = infer_signature(X_train_df, pipeline.predict(X_train_df))
        mlflow.sklearn.log_model(
            pipeline,
            artifact_path="model",
            signature=signature,
            input_example=X_train_df.head(5),
        )
    print(f"{run_name}: {run.info.run_id}")
    return run.info.run_id, metricas

baseline_id, baseline_metrics = entrenar_run_manual(
    "baseline_logreg",
    Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=5000, C=0.5)),
    ]),
)

improved_id, improved_metrics = entrenar_run_manual(
    "improved_logreg_balanced",
    Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=5000, C=2.0, class_weight="balanced")),
    ]),
)

2026/10/16 21:59:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


baseline_logreg: 415b67ad16754d69935fb97b88829621


2026/10/16 22:00:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


improved_logreg_balanced: cbbe13908fbd4844bd06ef16a1e5a79a


### 5.2 Autologging

In [6]:
mlflow.sklearn.autolog(log_input_examples=True, log_model_signatures=True, silent=True)

rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(
        n_estimators=300, max_depth=8, class_weight="balanced", random_state=42
    )),
])

with mlflow.start_run(run_name="autolog_random_forest") as run:
    rf_pipeline.fit(X_train_df, y_train)
    rf_metrics = calcular_metricas(rf_pipeline, X_test_df, y_test)
    mlflow.log_metrics(rf_metrics)
    mlflow.set_tags({"proyecto": TAG_PROYECTO, "tipo_logging": "autolog"})
    fig = figura_importancia(rf_pipeline, feature_names, "autolog_random_forest")
    mlflow.log_figure(fig, "graficas/importancia_variables.png")
    plt.close(fig)

rf_id = run.info.run_id
mlflow.sklearn.autolog(disable=True)
print(f"autolog_random_forest: {rf_id}")

D:\Documentos\GitHub\taller3_mlops\.venv\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(


autolog_random_forest: 8a5761cf82a34948a60152bde19af6f6


### 5.3 Comparación rápida


In [7]:
comparison = pd.DataFrame([
    {"run": "baseline_logreg", **baseline_metrics},
    {"run": "improved_logreg_balanced", **improved_metrics},
    {"run": "autolog_random_forest", **rf_metrics},
]).set_index("run").round(4)
display(comparison)

,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_balanced_accuracy,test_log_loss
run,,,,,,,
baseline_logreg,0.8167,0.7308,0.6264,0.6746,0.8913,0.7629,0.3749
improved_logreg_balanced,0.8200,0.6504,0.8791,0.7477,0.8896,0.8367,0.4066
autolog_random_forest,0.9133,0.8421,0.8791,0.8602,0.9590,0.9037,0.2954


## 6. Selección programática del mejor run

In [8]:
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_PATH],
    filter_string=f"tags.proyecto = '{TAG_PROYECTO}' and attributes.status = 'FINISHED'",
    order_by=[f"metrics.{METRICA_PRINCIPAL} DESC"],
)

columnas = [
    "run_id",
    "tags.mlflow.runName",
    "tags.tipo_logging",
    "metrics.test_f1",
    "metrics.test_recall",
    "metrics.test_roc_auc",
    "metrics.test_accuracy",
]
display(runs[columnas])

mejor_run = runs.iloc[0]
mejor_run_id = mejor_run["run_id"]
print(f"Mejor run: {mejor_run['tags.mlflow.runName']} ({mejor_run_id})")
print(f"{METRICA_PRINCIPAL} = {mejor_run[f'metrics.{METRICA_PRINCIPAL}']:.4f}")

,run_id,tags.mlflow.runName,tags.tipo_logging,metrics.test_f1,metrics.test_recall,metrics.test_roc_auc,metrics.test_accuracy
0,8a5761cf82a34948a60152bde19af6f6,autolog_random_forest,autolog,0.860215,0.879121,0.958988,0.913333
1,cbbe13908fbd4844bd06ef16a1e5a79a,improved_logreg_balanced,manual,0.747664,0.879121,0.889637,0.820000
2,415b67ad16754d69935fb97b88829621,baseline_logreg,manual,0.674556,0.626374,0.891267,0.816667


Mejor run: autolog_random_forest (8a5761cf82a34948a60152bde19af6f6)
test_f1 = 0.8602


## 7. MODELS: registro, alias `champion` y validación


In [9]:
version = mlflow.register_model(model_uri=f"runs:/{mejor_run_id}/model", name=MODEL_NAME)

client = MlflowClient()
client.set_registered_model_alias(MODEL_NAME, "champion", version.version)
print(f"Registrado {MODEL_NAME} versión {version.version} con alias 'champion'")

Successfully registered model 'workspace.default.taller3_mlops_model'.
2026/10/16 22:00:48 WARNING mlflow.tracking._model_registry.fluent: Run with id 8a5761cf82a34948a60152bde19af6f6 has no artifacts at artifact path 'model', registering model based on models:/m-93dd660947904403bcc68de9105beee9 instead


Created version '1' of model 'workspace.default.taller3_mlops_model'.


Registrado workspace.default.taller3_mlops_model versión 1 con alias 'champion'


In [10]:
modelo_champion = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
y_pred_champion = modelo_champion.predict(X_test_df)

validacion = {
    "accuracy": accuracy_score(y_test, y_pred_champion),
    "f1": f1_score(y_test, y_pred_champion),
    f"{METRICA_PRINCIPAL}_en_experimento": mejor_run[f"metrics.{METRICA_PRINCIPAL}"],
}
print("Validación del champion sobre test:", {k: round(v, 4) for k, v in validacion.items()})

Validación del champion sobre test: {'accuracy': 0.9133, 'f1': 0.8602, 'test_f1_en_experimento': np.float64(0.8602)}


## 8.Feature Store


In [11]:
USAR_FEATURE_STORE = False
FEATURE_TABLE = f"{CATALOGO}.{ESQUEMA}.taller3_mlops_features"

if USAR_FEATURE_STORE:
    try:
        from databricks.feature_engineering import FeatureEngineeringClient

        fe = FeatureEngineeringClient()
        features_pdf = pd.DataFrame(X, columns=feature_names)
        features_pdf.insert(0, "id", range(len(features_pdf)))
        features_sdf = spark.createDataFrame(features_pdf)

        if spark.catalog.tableExists(FEATURE_TABLE):
            fe.write_table(name=FEATURE_TABLE, df=features_sdf, mode="merge")
        else:
            fe.create_table(
                name=FEATURE_TABLE,
                primary_keys=["id"],
                df=features_sdf,
                description="Variables sintéticas del taller 3 de MLOps",
            )
        display(fe.read_table(name=FEATURE_TABLE).limit(5))
    except Exception as e:
        print("Feature Store no disponible, se omite esta sección:", e)
else:
    print("Feature Store desactivado (USAR_FEATURE_STORE = False).")

Feature Store desactivado (USAR_FEATURE_STORE = False).


## 9. Conclusiones

- **Trazabilidad:** en el notebook original las métricas solo quedaban impresas en la salida de la celda. Con MLflow cada entrenamiento queda guardado como un run con sus parámetros, métricas, gráficas y el modelo, y se puede comparar en la UI.
- **Manual vs. autolog:** autolog garantiza que no se pierde ningún hiperparámetro con muy poco código; el logging manual permite decidir exactamente qué métricas (de test) y artefactos se guardan. En la práctica conviene combinarlos.
- **Modelo reproducible:** al guardar el pipeline completo con firma e `input_example`, el escalado viaja junto con el modelo y MLflow valida el esquema de entrada al predecir.
- **Del experimento a producción:** `search_runs` elige el mejor modelo con un criterio explícito (F1 en test), el Model Registry en Unity Catalog lo versiona y el alias `champion` permite consumirlo sin depender del número de versión.